# GK-2A 12:00~14:00 Multi-year Pipeline

이 노트북은 **원본 수집을 전부 끝낸 뒤에만 tabular/long build를 수행**합니다.

- Phase 1: 2019~2025 원본 GK-2A + 14:00 ASOS 수집만 수행
- 기존 Drive 원본은 재다운로드하지 않음
- 완성된 연도는 수집 자체를 SKIP
- 일부만 받은 연도는 없는 파일만 이어받음
- Phase 2: 모든 연도 수집 완료 확인 후 tabular/long 변환 + TA/HM 병합
- Phase 3: 2019~2025 combined CSV 생성


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0. 설정


In [ ]:
from pathlib import Path

YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
START_MMDD = '08-24'
END_MMDD = '08-30'
START_TIME = '12:00'
END_TIME = '14:00'
STEP_MINUTES = 10

BRANCH = 'agent/shortterm-12to14-pipeline'
REPO_DIR = Path('/content/SME_DATA')
OUTPUT_ROOT = Path('/content/drive/MyDrive/SME_DATA/processed_station_features/shortterm_12to14_data')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('YEARS:', YEARS)
print('OUTPUT_ROOT:', OUTPUT_ROOT)


## 1. GitHub 최신화 + 패키지 설치


In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'fetch', 'origin'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
!pip -q install -e .
!pip -q install requests pandas numpy xarray h5netcdf netCDF4 pyyaml

print('현재 커밋:')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 2. KMA API Key 설정


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    key = userdata.get('KMA_API_KEY')
except Exception:
    key = None

if not key:
    key = getpass('KMA_API_KEY 입력: ').strip()
if not key:
    raise ValueError('KMA_API_KEY가 비어 있습니다.')

os.environ['KMA_API_KEY'] = key
print('KMA_API_KEY 설정 완료')


# Phase 1 — 원본 데이터만 전부 수집

이 셀에서는 **build를 절대 실행하지 않습니다.**

연도별로 현재 Drive 원본을 먼저 검사합니다.
- 원본이 모두 있으면 `[SKIP DOWNLOAD]`
- 일부만 있으면 `[RESUME DOWNLOAD]` 후 없는 파일만 다운로드

중간에 API 제한이나 Colab 종료가 발생해도 같은 셀을 다시 실행하면 이어받습니다.


In [ ]:
import sys, subprocess

cmd = [
    sys.executable, '-u', 'scripts/shortterm_multiyear_phased.py',
    '--phase', 'collect',
    '--years', *[str(y) for y in YEARS],
    '--start-mmdd', START_MMDD,
    '--end-mmdd', END_MMDD,
    '--start-time', START_TIME,
    '--end-time', END_TIME,
    '--step-minutes', str(STEP_MINUTES),
    '--output-root', str(OUTPUT_ROOT),
]

print('[PHASE 1] 원본 수집 시작')
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## Phase 1 상태 확인

정상 기준은 연도별 GK-2A **1,456/1,456**, ASOS **7/7**입니다.
`ALL_COLLECTION_COMPLETE=True`가 나온 뒤에만 Phase 2를 실행하세요.


In [ ]:
cmd = [
    sys.executable, '-u', 'scripts/shortterm_multiyear_phased.py',
    '--phase', 'status',
    '--years', *[str(y) for y in YEARS],
    '--start-mmdd', START_MMDD,
    '--end-mmdd', END_MMDD,
    '--start-time', START_TIME,
    '--end-time', END_TIME,
    '--step-minutes', str(STEP_MINUTES),
    '--output-root', str(OUTPUT_ROOT),
]
subprocess.run(cmd, cwd=REPO_DIR, check=True)


# Phase 2 — 모든 수집 완료 후 tabular/long 변환 + TA/HM 병합

이 단계는 **2019~2025 원본이 모두 완성됐는지 다시 검사**한 뒤 시작합니다.
하나라도 부족하면 자동으로 중단되며 build하지 않습니다.

모든 원본이 완성되면 2019~2025를 새로 build해서 과거 원본 누락이 tabular에 남지 않도록 합니다. 마지막에 combined CSV도 자동 생성합니다.


In [ ]:
cmd = [
    sys.executable, '-u', 'scripts/shortterm_multiyear_phased.py',
    '--phase', 'build',
    '--years', *[str(y) for y in YEARS],
    '--start-mmdd', START_MMDD,
    '--end-mmdd', END_MMDD,
    '--start-time', START_TIME,
    '--end-time', END_TIME,
    '--step-minutes', str(STEP_MINUTES),
    '--output-root', str(OUTPUT_ROOT),
]

print('[PHASE 2] tabular/long build 시작')
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## 최종 결과 확인


In [ ]:
import pandas as pd

COMBINED = OUTPUT_ROOT / 'datasets' / 'combined'
SUMMARY = COMBINED / 'shortterm_multiyear_summary.csv'
LONG = COMBINED / 'shortterm_long_2019to2025.csv'
WIDE = COMBINED / 'shortterm_wide_2019to2025.csv'
LABELS = COMBINED / 'shortterm_labels_1400_2019to2025.csv'

summary = pd.read_csv(SUMMARY)
display(summary)

long_df = pd.read_csv(LONG)
wide_df = pd.read_csv(WIDE)
labels_df = pd.read_csv(LABELS)

print('LONG:', long_df.shape)
print('WIDE:', wide_df.shape)
print('LABELS:', labels_df.shape)
print('Years:', sorted(wide_df['Year'].unique().tolist()))
print('Stations:', wide_df['STN_ID'].nunique())
print('TA missing:', labels_df['TA'].isna().sum())
print('HM missing:', labels_df['HM'].isna().sum())
